In [11]:
import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt

Load the Dataset

In [12]:
df = pd.read_csv("processed_telco_churn.csv")
from sklearn.model_selection import train_test_split
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [13]:
best_lr=joblib.load('../models/customer_churn_model.pkl')

Load the Trained Model

In [14]:
logistic_model = best_lr.named_steps["model"]

In [15]:
scaler = best_lr.named_steps["scaler"]

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [16]:

explainer = shap.LinearExplainer(
    logistic_model,
    X_train_scaled
)

Background dataset has 5634 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=5634 when initializing the masker.


In [17]:
shap_values = explainer.shap_values(X_test_scaled)

In [23]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=X_test.columns,show=False
)
plt.savefig("../plots/shap plots/shap_summary_plot.png",
            dpi=300,
            bbox_inches="tight")

plt.close()

In [24]:
customer_index=0
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[customer_index],
        base_values=explainer.expected_value,
        data=X_test.iloc[customer_index],
        feature_names=X_test.columns,
        
    ),show=False
   

)



plt.savefig(
    "../plots/shap plots/shap_waterfall_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

